# Experiment 1, follow-up: added value, the over-budget confound, and the live pre-screen

Reads the outputs of `scripts/06_incremental_value.py` (on `main`) and `scripts/07_prescreen_analysis.py` (on `prescreen`).

**How to read the scores.** All predictions are out-of-fold with leave-one-run-out cross-validation. The probe layer is chosen inside each training split, so the scores carry no layer-selection bias.
Models: **M** = metadata (operator, generation, parent fitness / failure / over-budget status, prompt length); **T** = text (code TF-IDF, entropy, log-prob, length); **P** = activation probe. Intervals are 95% bootstrap intervals over LLaMEA runs.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

MI_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(MI_ROOT))
from common.paths import data_dir, results_dir
from common.plotting import INK, NEUTRAL, POSITION_COLORS, POSITION_LABELS, SERIES, setup

setup()
EXP = "exp01_probing"
TAG = "main"
PRESCREEN_TAG = "prescreen"
fu = results_dir(EXP, TAG) / "followup"
inc = pd.read_csv(fu / "incremental.csv")
strata = pd.read_csv(fu / "strata.csv")

## 1. Do the activations add information beyond metadata and text?

In [ ]:
scores = inc[~inc["model"].str.contains(" - ")].pivot_table(
    index=["target", "position"], columns="model", values="score"
)[["M", "M+T", "P", "M+P", "M+T+P"]]
scores.round(3)

In [ ]:
deltas = inc[inc["model"].isin(["M+P - M", "M+T+P - M+T"])].copy()
order = ["child_failed", "over_budget", "improved", "child_fitness"]
positions = ["last", "prompt_mean", "gen_mean"]
rows = [(t, p) for t in order for p in positions]
fig, ax = plt.subplots(figsize=(7, 5.6))
for k, (comp, color, off) in enumerate(
    [("M+P - M", SERIES[0], -0.15), ("M+T+P - M+T", SERIES[1], 0.15)]
):
    d = deltas[deltas["model"] == comp].set_index(["target", "position"])
    for i, key in enumerate(rows):
        r = d.loc[key]
        ax.errorbar(
            r["score"], i + off,
            xerr=[[r["score"] - r["ci_low"]], [r["ci_high"] - r["score"]]],
            fmt="o", color=color, ms=5, lw=1.5, capsize=0,
            label=("probe added to metadata" if comp == "M+P - M" else
                   "probe added to metadata + text") if i == 0 else None,
        )
ax.axvline(0, color=NEUTRAL, lw=1)
ax.set_yticks(range(len(rows)))
ax.set_yticklabels([f"{t} · {POSITION_LABELS[p]}" for t, p in rows])
ax.invert_yaxis()
for i in range(3, len(rows), 3):
    ax.axhline(i - 0.5, color="#e4e3df", lw=0.8)
ax.grid(axis="y", visible=False)
ax.set_xlabel("gain from adding the probe (AUROC or Spearman ρ)")
ax.set_title("Added value of the activations, out-of-fold, nested layer choice", loc="left", color=INK)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(results_dir(EXP, TAG) / "incremental_value.png", dpi=150)

## 2. Over-budget: is the signal inherited from the parent?

Of the working children whose parent overspent, 99% overspend too, so the parent's status alone is a strong predictor (it is part of **M** here).
Within a stratum the parent's status is constant and cannot explain the prediction. With 773 of 780 positives, the "parent over budget" stratum is not testable.

In [ ]:
strata.round(3)

In [ ]:
testable = ["parent within budget", "no working parent"]
series = [("P", p, POSITION_COLORS[p], f"probe, {POSITION_LABELS[p]}") for p in positions]
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharex=True)
for ax, stratum in zip(axes, testable):
    s = strata[strata["stratum"] == stratum].set_index("position")
    n, pos_n = s.iloc[0]["n"], s.iloc[0]["positives"]
    items = [(f"{lbl}", s.loc[p, "P"], s.loc[p, "P_ci_low"], s.loc[p, "P_ci_high"], c)
             for _, p, c, lbl in series]
    g = s.loc["gen_mean"]
    items.append(("metadata + text", g["M+T"], g["M+T_ci_low"], g["M+T_ci_high"], NEUTRAL))
    for i, (lbl, v, lo, hi, c) in enumerate(items):
        ax.errorbar(v, i, xerr=[[v - lo], [hi - v]], fmt="o", color=c, ms=6, lw=1.5)
    ax.set_yticks(range(len(items)))
    ax.set_yticklabels([it[0] for it in items] if ax is axes[0] else [])
    ax.invert_yaxis()
    ax.axvline(0.5, color=NEUTRAL, lw=1, ls="--")
    ax.grid(axis="y", visible=False)
    ax.set_title(f"{stratum} (n={n}, {pos_n} overspend)", loc="left", fontsize=10, color=INK)
    ax.set_xlabel("AUROC for over_budget")
fig.tight_layout()
fig.savefig(results_dir(EXP, TAG) / "over_budget_strata.png", dpi=150)

## 3. Live pre-screen on new seeds (`prescreen`)

Probe and threshold frozen from `main` (`child_failed`, generated-code mean, layer 21, threshold = 80th percentile of the out-of-fold scores on `main`).
Seeds 100-111, each run twice: **shadow** (LLaMEA sees everything) and **screened** (flagged code is shown as failed). Every algorithm is still evaluated, so the true label is known for all.

In [ ]:
pres = results_dir(EXP, PRESCREEN_TAG)
pmeta = pd.read_csv(data_dir(EXP, "processed", PRESCREEN_TAG) / "meta.csv", dtype={"record_id": str})
oof_main = pd.read_csv(results_dir(EXP, TAG) / "oof_child_failed_gen_mean.csv")
threshold = pd.read_json(results_dir(EXP, TAG) / "probes" / "probes.json")["child_failed_gen_mean_L21"]["threshold"]
pd.read_csv(pres / "screen_metrics.csv").round(3)

In [ ]:
def tradeoff(y, score):
    order = np.argsort(-score)
    y = np.asarray(y)[order]
    k = np.arange(1, len(y) + 1)
    return k / len(y), np.cumsum(y) / y.sum(), np.cumsum(1 - y) / (1 - y).sum()

fig, ax = plt.subplots(figsize=(6.5, 4))
curves = [
    ("main, out-of-fold", oof_main["child_failed"], oof_main["pred"], "--"),
    ("new seeds, frozen probe", pmeta["child_failed"], pmeta["prescreen_score"], "-"),
]
for lbl, y, s, ls in curves:
    f, caught, lost = tradeoff(y, np.asarray(s))
    ax.plot(f, caught, color=SERIES[2], ls=ls, label=f"failures caught ({lbl})")
    ax.plot(f, lost, color=SERIES[1], ls=ls, label=f"working algorithms lost ({lbl})")
flag_rate = (pmeta["prescreen_score"] > threshold).mean()
ax.axvline(flag_rate, color=NEUTRAL, lw=1)
ax.annotate(f"frozen threshold\nflags {flag_rate:.0%} on new seeds", (flag_rate, 0.05),
            xytext=(6, 0), textcoords="offset points", color=INK, fontsize=9)
ax.plot([0, 1], [0, 1], color=NEUTRAL, lw=1, ls=":")
ax.set_xlabel("fraction of evaluations skipped (highest probe score first)")
ax.set_ylabel("fraction")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(fontsize=8, loc="lower right")
ax.set_title("Pre-screen trade-off generalises to new seeds", loc="left", color=INK)
fig.tight_layout()
fig.savefig(pres / "prescreen_heldout.png", dpi=150)

### Effect on the search (paired by seed)

In [ ]:
runs = pd.read_csv(pres / "runs.csv")
runs.pivot(index="seed", columns="condition", values=["best", "failure_rate", "flagged"]).round(3)

In [ ]:
bsf = pd.read_csv(pres / "best_so_far.csv")
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for cond, color in [("shadow", SERIES[0]), ("screened", SERIES[1])]:
    d = bsf[bsf["condition"] == cond].pivot_table(index="eval_idx", columns="seed", values="best_so_far")
    d = d.ffill()
    med, lo, hi = d.median(axis=1), d.quantile(0.25, axis=1), d.quantile(0.75, axis=1)
    ax.plot(med.index, med, color=color, label=f"{cond} (median, IQR over seeds)")
    ax.fill_between(med.index, lo, hi, color=color, alpha=0.15, lw=0)
ax.set_xlabel("generated algorithms")
ax.set_ylabel("best fitness LLaMEA can select (AOCC)")
ax.legend(loc="lower right")
ax.set_title("Search progress with and without acting on the pre-screen", loc="left", color=INK)
fig.tight_layout()
fig.savefig(pres / "prescreen_search.png", dpi=150)